# NumCompute Quickstart Demo

This notebook demonstrates the main features of the NumCompute toolkit using only **plain Python and NumPy**. It is designed to support the Assignment 2.1 demo requirement by showing CSV loading, preprocessing, sorting/searching, ranking, statistics, gradients/Jacobian, metrics, pipeline usage, and vectorised-vs-loop benchmarks.

## 1. Setup and imports

This cell makes the notebook work whether it is opened from the project root or from inside the `demo/` folder.

In [45]:
from pathlib import Path
import sys
import time
import platform

import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "numcompute").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from numcompute.io import IO
from numcompute.preprocessing import SimpleImputer, StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.sort_search import topk, quickselect, binary_search
from numcompute.ranks import rank, percentile, percentiles
from numcompute.stats import Stats, StreamingStats
from numcompute.metrics import Classification, Regression
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline
from numcompute.utils import sigmoid, softmax, logsumexp, euclidean_distance, cosine_similarity

np.set_printoptions(precision=3, suppress=True)
print("Project root:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)

Project root: /mnt/d/Adelaide University/Semester 1/Programming for AI 1/Assignments/New folder/NumCompute
Python: 3.14.3
NumPy: 2.4.3


## 2. Read CSV data using `io.py`

The CSV file contains numeric values, categorical values, booleans, and missing values. This demonstrates dtype inference and missing-value handling.

In [52]:
csv_path = PROJECT_ROOT / "demo" / "numcompute_demo_data.csv"
csv_data = IO.load_csv(str(csv_path))

print(csv_data)
print("Data array:")
print(csv_data.data)
print("Column metadata:")
for col in csv_data.cols:
    print(f"{col.name}: {col.dtype}")

Shape: (30, 13)
Headers: id: int, age: int, study_hours: float, attendance_pct: int, previous_score: int, city: str, study_mode: str, assignment_score: int, exam_score: int, final_score: int, passed_true: int, passed_pred: int, pass_prob: float
Data array:
[[1.0 20.0 12.5 88.0 72.0 'Adelaide' 'Online' 68.0 74.0 72.0 1.0 1.0 0.82]
 [2.0 21.0 10.0 91.0 72.0 'Sydney' 'OnCampus' 70.0 71.0 72.0 1.0 1.0 0.79]
 [3.0 22.0 8.0 75.0 65.0 'Melbourne' 'Hybrid' 60.0 66.0 64.0 1.0 0.0 0.49]
 [4.0 19.0 6.5 60.0 55.0 'Adelaide' 'Online' 58.0 54.0 55.0 0.0 0.0 0.28]
 [5.0 23.0 nan 85.0 80.0 'Perth' 'OnCampus' 77.0 81.0 80.0 1.0 1.0 0.88]
 [6.0 24.0 14.0 96.0 88.0 'Brisbane' 'Hybrid' 84.0 90.0 88.0 1.0 1.0 0.95]
 [7.0 20.0 7.5 70.0 61.0 'Sydney' 'Online' 62.0 60.0 61.0 1.0 0.0 0.47]
 [8.0 21.0 5.0 58.0 49.0 'Adelaide' 'OnCampus' 50.0 48.0 49.0 0.0 0.0
  0.19]
 [9.0 22.0 11.0 83.0 77.0 'Melbourne' nan 78.0 76.0 77.0 1.0 1.0 0.84]
 [10.0 23.0 9.0 79.0 69.0 'Perth' 'Online' 70.0 68.0 69.0 1.0 1.0 0.72]
 [1

## 3. Split numeric and categorical features

The preprocessing module uses numeric transformations for numeric columns and one-hot encoding for categorical columns.

In [47]:
numeric_idx = [i for i, col in enumerate(csv_data.cols) if col.dtype in ("int", "float")]
categorical_idx = [i for i, col in enumerate(csv_data.cols) if col.dtype == "str"]

X_numeric = csv_data.data[:, numeric_idx].astype(float)
X_category = csv_data.data[:, categorical_idx].astype(object)

print("Numeric feature matrix:")
print(X_numeric)
print("Categorical feature matrix:")
print(X_category)

Numeric feature matrix:
[[  1.     20.     12.5    88.     72.     68.     74.     72.      1.
    1.      0.82 ]
 [  2.     21.     10.     91.     72.     70.     71.     72.      1.
    1.      0.79 ]
 [  3.     22.      8.     75.     65.     60.     66.     64.      1.
    0.      0.49 ]
 [  4.     19.      6.5    60.     55.     58.     54.     55.      0.
    0.      0.28 ]
 [  5.     23.        nan  85.     80.     77.     81.     80.      1.
    1.      0.88 ]
 [  6.     24.     14.     96.     88.     84.     90.     88.      1.
    1.      0.95 ]
 [  7.     20.      7.5    70.     61.     62.     60.     61.      1.
    0.      0.47 ]
 [  8.     21.      5.     58.     49.     50.     48.     49.      0.
    0.      0.19 ]
 [  9.     22.     11.     83.     77.     78.     76.     77.      1.
    1.      0.84 ]
 [ 10.     23.      9.     79.     69.     70.     68.     69.      1.
    1.      0.72 ]
 [ 11.     19.      4.5    55.     45.     44.     46.     45.      0.
    0

## 4. Preprocessing with imputation, scaling, and one-hot encoding

This section demonstrates the preprocessing components: missing-value imputation, standard scaling, min-max scaling, and categorical one-hot encoding.

In [48]:
imputer = SimpleImputer(strategy="mean")
X_filled = imputer.fit_transform(X_numeric)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)

minmax = MinMaxScaler()
X_minmax = minmax.fit_transform(X_filled)

encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X_category)

X_final = np.hstack([X_scaled, X_encoded])

print("After mean imputation:")
print(X_filled)
print("After standard scaling:")
print(X_scaled)
print("After min-max scaling:")
print(X_minmax)
print("One-hot encoded category columns:")
print(X_encoded)
print("Final combined feature matrix shape:", X_final.shape)
print(X_final)

After mean imputation:
[[  1.     20.     12.5    88.     72.     68.     74.     72.      1.
    1.      0.82 ]
 [  2.     21.     10.     91.     72.     70.     71.     72.      1.
    1.      0.79 ]
 [  3.     22.      8.     75.     65.     60.     66.     64.      1.
    0.      0.49 ]
 [  4.     19.      6.5    60.     55.     58.     54.     55.      0.
    0.      0.28 ]
 [  5.     23.      8.966  85.     80.     77.     81.     80.      1.
    1.      0.88 ]
 [  6.     24.     14.     96.     88.     84.     90.     88.      1.
    1.      0.95 ]
 [  7.     20.      7.5    70.     61.     62.     60.     61.      1.
    0.      0.47 ]
 [  8.     21.      5.     58.     49.     50.     48.     49.      0.
    0.      0.19 ]
 [  9.     22.     11.     83.     77.     78.     76.     77.      1.
    1.      0.84 ]
 [ 10.     23.      9.     79.     69.     70.     68.     69.      1.
    1.      0.72 ]
 [ 11.     19.      4.5    55.     45.     44.     46.     45.      0.
    0.

## 5. Pipeline abstraction

The lightweight `Pipeline` class allows reusable transformations to be chained together with a consistent `fit_transform` workflow.

In [49]:
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

X_pipeline = pipe.fit_transform(X_numeric)
print("Pipeline output:")
print(X_pipeline)

Pipeline output:
[[-1.675 -0.989  0.922  0.728  0.336  0.183  0.443  0.356  0.707  0.761
   0.745]
 [-1.56  -0.503  0.27   0.896  0.336  0.3    0.264  0.356  0.707  0.761
   0.637]
 [-1.444 -0.017 -0.252  0.004 -0.08  -0.285 -0.036 -0.116  0.707 -1.314
  -0.436]
 [-1.329 -1.475 -0.643 -0.832 -0.673 -0.402 -0.755 -0.647 -1.414 -1.314
  -1.187]
 [-1.213  0.469  0.     0.561  0.811  0.71   0.863  0.828  0.707  0.761
   0.959]
 [-1.098  0.955  1.314  1.174  1.286  1.119  1.402  1.3    0.707  0.761
   1.21 ]
 [-0.982 -0.989 -0.382 -0.275 -0.317 -0.168 -0.396 -0.293  0.707 -1.314
  -0.507]
 [-0.867 -0.503 -1.035 -0.944 -1.03  -0.869 -1.115 -1.001 -1.414 -1.314
  -1.509]
 [-0.751 -0.017  0.531  0.45   0.633  0.768  0.563  0.651  0.707  0.761
   0.816]
 [-0.635  0.469  0.009  0.227  0.158  0.3    0.084  0.179  0.707  0.761
   0.387]
 [-0.52  -1.475 -1.165 -1.111 -1.267 -1.22  -1.235 -1.237 -1.414 -1.314
  -1.688]
 [-0.404  1.442  1.575  1.286  1.464  1.47   1.522  1.477  0.707  0.761
   1.317]

## 6. Sorting, top-k, quickselect, and binary search

This section demonstrates algorithmic components from `sort_search.py`.

In [51]:
scores = csv_data.data[:, 9].astype(float)
scores_clean = scores[~np.isnan(scores)]

print("Scores without NaN:", scores_clean)

top_values, top_indices = topk(scores_clean, k=2, largest=True, return_indices=True)
print("Top 2 values:", top_values)
print("Top 2 indices:", top_indices)

sorted_scores = np.sort(scores_clean)
print("Sorted scores:", sorted_scores)

index, found = binary_search(sorted_scores, 88.0)
print("Binary search for 88.0 -> index:", index, "found:", found)

third_smallest = quickselect(scores_clean, k=2)
print("3rd smallest score using quickselect:", third_smallest)

Scores without NaN: [72. 72. 64. 55. 80. 88. 61. 49. 77. 69. 45. 91. 65. 65. 65. 38. 94. 72.
 30. 72. 82. 67. 25. 96. 56. 78. 59. 59. 59. 74.]
Top 2 values: [96. 94.]
Top 2 indices: [23 16]
Sorted scores: [25. 30. 38. 45. 49. 55. 56. 59. 59. 59. 61. 64. 65. 65. 65. 67. 69. 72.
 72. 72. 72. 74. 77. 78. 80. 82. 88. 91. 94. 96.]
Binary search for 88.0 -> index: 26 found: True
3rd smallest score using quickselect: 38.0


## 7. Ranking and percentiles

Ranking demonstrates tie handling. Percentiles provide useful descriptive summaries of numerical arrays.

In [53]:
print("Average ranks:", rank(scores_clean, method="average"))
print("Dense ranks:", rank(scores_clean, method="dense"))
print("Ordinal ranks:", rank(scores_clean, method="ordinal"))

print("50th percentile:", percentile(scores_clean, 50))
print("25th, 50th, 75th percentiles:", percentiles(scores_clean, [25, 50, 75]))

Average ranks: [18.5 18.5 11.   5.  24.  26.  10.   4.  22.  16.   3.  27.  13.  13.
 13.   2.  28.  18.5  1.  18.5 25.  15.   0.  29.   6.  23.   8.   8.
  8.  21. ]
Dense ranks: [13. 13.  9.  5. 17. 19.  8.  4. 15. 12.  3. 20. 10. 10. 10.  2. 21. 13.
  1. 13. 18. 11.  0. 22.  6. 16.  7.  7.  7. 14.]
Ordinal ranks: [17 18 11  5 24 26 10  4 22 16  3 27 12 13 14  2 28 19  1 20 25 15  0 29
  6 23  7  8  9 21]
50th percentile: 66.0
25th, 50th, 75th percentiles: [59.   66.   76.25]


## 8. Descriptive and streaming statistics

`Stats` provides column-wise summaries, histograms, quantiles, and streaming mean/variance using Welford's algorithm.

In [54]:
print("Column-wise means:")
print(Stats.mean(csv_data, axis=0))

print("Column-wise standard deviations:")
print(Stats.std(csv_data, axis=0))

print("Column-wise median/0.5 quantile:")
print(Stats.quantile(csv_data, 0.5, axis=0))

counts, edges = Stats.histogram(csv_data, bins=4)
print("Histogram counts:", counts)
print("Histogram bin edges:", edges)

print("Streaming mean over all numeric values:", Stats.streaming_mean(csv_data))
print("Streaming variance over all numeric values:", Stats.streaming_variance(csv_data, ddof=0))

stream = StreamingStats()
stream.update_many([1, 2, 3, np.nan, 4, 5])
print("Manual StreamingStats mean:", stream.mean)
print("Manual StreamingStats variance:", stream.variance(ddof=0))

Column-wise means:
{'id': np.float64(15.5), 'age': np.float64(22.03448275862069), 'study_hours': np.float64(8.96551724137931), 'attendance_pct': np.float64(74.93103448275862), 'previous_score': np.float64(66.34482758620689), 'assignment_score': np.float64(64.86666666666666), 'exam_score': np.float64(66.6), 'final_score': np.float64(65.96666666666667), 'passed_true': np.float64(0.6666666666666666), 'passed_pred': np.float64(0.6333333333333333), 'pass_prob': np.float64(0.6118333333333335)}
Column-wise standard deviations:
{'id': np.float64(8.65544144839919), 'age': np.float64(2.0923960629349057), 'study_hours': np.float64(3.8972382672569172), 'attendance_pct': np.float64(18.247212417210775), 'previous_score': np.float64(17.132900114971303), 'assignment_score': np.float64(17.101137064209762), 'exam_score': np.float64(16.68652150689292), 'final_score': np.float64(16.949893477213617), 'passed_true': np.float64(0.4714045207910317), 'passed_pred': np.float64(0.48189440982669857), 'pass_prob':

## 9. Classification and regression metrics

This section demonstrates binary classification metrics and mean squared error.

In [55]:
y_true = np.array([1, 0, 1, 1, 0, 1])
y_pred = np.array([1, 0, 0, 1, 0, 1])

print("Confusion matrix (tp, tn, fp, fn):", Classification.confusion_matrix(y_true, y_pred))
print("Accuracy:", Classification.accuracy(y_true, y_pred))
print("Precision:", Classification.precision(y_true, y_pred))
print("Recall:", Classification.recall(y_true, y_pred))
print("F1:", Classification.f1(y_true, y_pred))

actual = np.array([82.0, 76.0, 91.0, 88.0])
predicted = np.array([80.0, 78.0, 90.0, 89.0])
print("MSE:", Regression.mse(actual, predicted))

Confusion matrix (tp, tn, fp, fn): (np.int64(3), np.int64(2), np.int64(0), np.int64(1))
Accuracy: 0.8333333333333334
Precision: 1.0
Recall: 0.75
F1: 0.8571428571428571
MSE: 2.5


## 10. Finite-difference gradient and Jacobian

`optim.py` estimates gradients for scalar functions and Jacobians for vector-valued functions using finite differences.

In [56]:
def objective(x):
    return x[0] ** 2 + 3 * x[1] ** 2 + 4 * x[2]

x0 = np.array([2.0, 3.0, 4.0])
print("Gradient at", x0, ":", grad(objective, x0, method="central"))


def vector_function(x):
    return np.array([
        x[0] + x[1] + x[2],
        x[0] ** 2,
        np.sin(x[1]),
    ])

print("Jacobian:")
print(jacobian(vector_function, x0, method="central"))

Gradient at [2. 3. 4.] : [ 4. 18.  4.]
Jacobian:
[[ 1.    1.    1.  ]
 [ 4.    0.    0.  ]
 [ 0.   -0.99  0.  ]]


## 11. Numerical utilities

The utility module includes stable activation functions, logsumexp, distances, similarity, and batching helpers.

In [57]:
large_values = np.array([-1000.0, 0.0, 1000.0])
print("Stable sigmoid:", sigmoid(large_values))

logits = np.array([1000.0, 1001.0, 1002.0])
print("Stable softmax:", softmax(logits))
print("Stable logsumexp:", logsumexp(logits))
print("All -inf logsumexp:", logsumexp(np.array([-np.inf, -np.inf])))

print("Euclidean distance:", euclidean_distance([1, 2, 3], [2, 2, 4]))
print("Cosine similarity:", cosine_similarity([1, 0, 1], [0, 1, 1]))

Stable sigmoid: [0.  0.5 1. ]
Stable softmax: [0.09  0.245 0.665]
Stable logsumexp: 1002.4076059644444
All -inf logsumexp: -inf
Euclidean distance: 1.4142135623730951
Cosine similarity: 0.4999999999999999


## 12. Benchmark: vectorised NumPy vs Python loops

This cell performs a small benchmark directly inside the notebook. The standalone version is available at `benchmark/run_benchmarks.py`.

In [58]:
def loop_mean(values):
    total = 0.0
    count = 0
    for value in values:
        total += float(value)
        count += 1
    return total / count


def loop_mse(y_true, y_pred):
    total = 0.0
    count = 0
    for a, b in zip(y_true, y_pred):
        err = a - b
        total += err * err
        count += 1
    return total / count


def time_call(func, *args, repeats=5):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        func(*args)
        times.append(time.perf_counter() - start)
    return min(times)

rng = np.random.default_rng(42)
x = rng.normal(size=100_000)
y_true_b = rng.normal(size=100_000)
y_pred_b = y_true_b + rng.normal(scale=0.1, size=100_000)

benchmarks = [
    ("Mean", lambda arr: np.mean(arr), loop_mean, (x,)),
    ("MSE", lambda a, b: np.mean((a - b) ** 2), loop_mse, (y_true_b, y_pred_b)),
]

print(f"{'Task':<10} {'Vectorised (s)':<16} {'Loop (s)':<12} {'Speedup':<10}")
print("-" * 52)
for name, vec_func, loop_func, args in benchmarks:
    vec_time = time_call(vec_func, *args)
    loop_time = time_call(loop_func, *args)
    speedup = loop_time / vec_time if vec_time > 0 else np.inf
    print(f"{name:<10} {vec_time:<16.6f} {loop_time:<12.6f} {speedup:<10.2f}x")

Task       Vectorised (s)   Loop (s)     Speedup   
----------------------------------------------------
Mean       0.000022         0.009052     420.56    x
MSE        0.000088         0.016835     190.90    x


## 13. Running tests

The command below is commented out so the notebook does not unexpectedly run a full test suite. Uncomment it when running from the project root if `pytest` is installed.

In [59]:
!python -m pytest -q "$PROJECT_ROOT"/"tests"/*

........................................................................ [ 43%]
........................................................................ [ 86%]
......................                                                   [100%]
=============================== warnings summary ===============================
tests/test_metrics.py::test_empty_input
  /home/pausage/.pyenv/versions/ml_workshop/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
    return _methods._mean(a, axis=axis, dtype=dtype,

tests/test_metrics.py::test_empty_input
  /home/pausage/.pyenv/versions/ml_workshop/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
    ret = ret.dtype.type(ret / rcount)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
166 passed, 2 warnings in 1.34s


## 14. Demo summary

This quickstart demonstrates the main assignment requirements:

- CSV loading with missing values
- preprocessing with imputation, scaling, and one-hot encoding
- pipeline abstraction
- top-k, quickselect, and binary search
- ranking and percentiles
- descriptive statistics, histograms, quantiles, and Welford streaming stats
- classification and regression metrics
- finite-difference gradient and Jacobian
- stable numerical utilities
- benchmark comparison between vectorised NumPy and Python loops